# Requirement 19: Resource limits

> Enforce execution timeout and limits on CPU, memory, disk, and output.

**Covered by:** Both

**How it is fulfilled:** The platform fixes CPU, memory and disk per sandbox. The orchestrator enforces a time limit with `timeout` and caps output size before returning results.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
info = sandbox.get()
print("resources:", info.resources if hasattr(info, "resources") else "see stats below")
print(sandbox.get_stats())
print(sandbox.exec("nproc; free -m | head -2; df -h / | tail -1").stdout)

resources: SandboxResources(cpu='1000m', memory='2048Mi', disk='20480Mi')
SandboxStats(cpu=CpuUsage(usage_nano_cores=None, usage_core_nano_seconds=None), memory=ResourceUsage(used_bytes=219025408, total_bytes=2326118400, available_bytes=2107092992), storage=None, network=NetworkUsage(rx_bytes=0, tx_bytes=0))


1
               total        used        free      shared  buff/cache   available
Mem:            2218         218        1988           0         143        1999
/dev/vdb         20G  7.0G   12G  38% /



In [3]:
# Timeout: a runaway loop is killed after 5 seconds
run = sandbox.exec("timeout 5 python -c 'while True: pass'")
print("exit code:", run.exit_code)
check("runaway code is killed by the time limit (exit 124)", run.exit_code == 124)

exit code: 124
PASS  runaway code is killed by the time limit (exit 124)


In [4]:
# Output cap: the orchestrator truncates what it returns
MAX_OUTPUT = 1000
run = sandbox.exec("python -c \"print('x' * 100000)\"")
returned = run.stdout[:MAX_OUTPUT]
print(f"sandbox produced {len(run.stdout)} chars; orchestrator returns {len(returned)}")
check("output is capped", len(returned) == MAX_OUTPUT)
cleanup(sandbox)

sandbox produced 100001 chars; orchestrator returns 1000
PASS  output is capped


**Caveat:** CPU, memory and disk are sized when the sandbox is created and cannot be changed by the generated code. The time limit and output cap are the orchestrator's own logic.